<a href="https://colab.research.google.com/github/deltorobarba/science/blob/main/gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Gemma 4 - Finetuning (PEFT)**

*Alexander Del Toro Barba (PhD), Machine Learning and Quantum Computing Specialist at Google Cloud*

This tutorial describes how to load Gemma 4 from Huggingface Hub, fine-tune it locally on a GPU and run an inference. Please note that you can finetune Gemma 4 easier using tools like [unsloth](https://huggingface.co/collections/unsloth/gemma-4) or [axolotl](https://huggingface.co/axolotl-ai-co/models). But they hide many components under the hood. Here we want to dive deeper and run fine-tuning on a lower level.

> **I explained and commented everything in detail so that both beginners and advanced users can understand and work with it** 😇 OK, let's start.. 🚀



👉 We will use model_id = "**google/gemma-4-E4B-it**". The "E" in E4B stands for "effective" (effective parameters). Although the model has approximately 8 billion parameters in total (including embeddings), it behaves like a 4.5 billion parameter model.

**Gemma 4 has two new, important features: AutoProcessor for multimodality and the Thinking instruct.**

* ⚠️️ **`Thinking mode`**: is the new reasoning mode in Gemma 4: https://ai.google.dev/gemma/docs/capabilities/thinking. If you want you use the simpler HuggingFace 'pipeline()' API, you can't use Thinking mode, because the API doesn't expose thinking mode easily. You need to move to the lower-level 'AutoProcessor' approach to enable it during inference.

* ⚠️ **`AutoProcessor`**: The old Gemma 1-3 were text-only, where the input handler is **'AutoTokenizer'** (text only). But new Gemma 4 is multimodal by default and requires a multimodal input handler - the **'AutoProcessor'**. It handles the new **'mm_token_type_ids'** which is a tensor that the model expects on every call to distinguish text tokens from image tokens, even when there's no image. 'AutoProcessor' is a separate class that converts raw inputs (text, images) into tensor format the model expects (inside 'pipeline()').

**Problem: Missing `Gemma4ClippableLinear` support in PEFT (LoraConfig) 💥**

* Multimodality is also the reason why we can not directly use the [Gemma 3 fine-tuning tutorial](https://ai.google.dev/gemma/docs/core/huggingface_vision_finetune_qlora) and just switch the model 🥶

* We can also not use 'exclude_modules' or Regex approach within `lora_config = LoraConfig()` (see comments in code) 😭

* Every layer inside Gemma 3 is a standard `torch.nn.Linear`. PEFT knows how to scan for those, wrap them with LoRA matrices, and freeze the rest. The vision encoder in Gemma 3 uses the same standard layer type as the language model — so PEFT can treat the whole model uniformly.

* ⚠️ **`Gemma4ClippableLinear`**: But Gemma 3 was not a native multi-modal model like Gemma 4. What changed in Gemma 4 is the `Gemma4ClippableLinear`, a new custom layer type that Google introduced. It wraps the standard linear layer inside the vision and audio encoders, adding the ability to "clip" (prune) weights for efficiency. It like as if `torch.nn.Linear` wearing a custom jacket that PEFT doesn't recognise.

  ```python
  model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)

  peft_config = LoraConfig(target_modules="all-linear", ...)

  trainer = SFTTrainer(model=model, peft_config=peft_config, ...)
  ```

* When PEFT scans the model to apply LoRA, it encounters these layers and crashes, because it only knows how to wrap layer types it has been explicitly programmed to handle. **`Gemma4ClippableLinear` was released so recently that PEFT simply hasn't added support for it yet.** 💡 This is why we can not simply take the gemma 3 finetuning code and instead need a workaround.


⚠️ **Solution: Since Gemma 4 is multimodal where its vision and audio encoder uses a custom 'Gemma4ClippableLinear' wrapper that PEFT doesn't simply recognize, we need to do following if we want to properly fine-tune Gemma 4**:

1. Add a `prepare_model_for_kbit_training` step before LoRA on the quantized model.
2. Exclude unused vision and audio encoder and only finetune the language model layers.

The step `prepare_model_for_kbit_training` is actually independent of the Gemma 4 problem, but it helps us here. It's needed any time we load a model in 4-bit quantization (which we do via `BitsAndBytesConfig`). It does three things:

  ```
  1. Freezes all quantized weights       → marks them as non-trainable
  2. Casts normalisation layers to fp32  → LayerNorm etc. must stay in full precision or gradients become NaN
  3. Enables gradient checkpointing      → trades compute for VRAM by recomputing activations on the backward pass
                                          instead of storing them all
  ```

* Gemma 3 needs this too. In this Gemma 4 tutorial we just hide it inside the `SFTTrainer` when we pass `peft_config`. Because we have to apply LoRA manually (due to the ClippableLinear problem), we have to call it explicitly ourself first. The core insight is that **PEFT's problem is only with the vision/audio towers. The language model layers are perfectly normal.** So the trick is to temporarily hide the audio and vision towers from PEFT:

  ```
  │ Gemma4Model                                         
  │                                                     
  │  vision_tower  ← Gemma4ClippableLinear  💥 PEFT     
  │  audio_tower   ← Gemma4ClippableLinear  💥 PEFT     
  │  embed_vision  ← Gemma4ClippableLinear  💥 PEFT     
  │  embed_audio   ← Gemma4ClippableLinear  💥 PEFT     
  │                                                     
  │  language_model ← standard nn.Linear    ✅ PEFT      
  ```

* **In step 1 we save & swap:** the `Identity()` is the simplest possible PyTorch module. It just returns its input unchanged. Since it is a standard recognised type, PEFT scans past it without complaint. The real tower is safe in the Python variable, still in GPU memory.

  ```python
  vision_tower = model.model.vision_tower         # we save real tower to a variable
  model.model.vision_tower = torch.nn.Identity()  # we replace with a dummy
  ```

* **In step 2 we let PEFT wrap the model**: PEFT scans the model, finds only standard `nn.Linear` layers (in the language model), and wraps those with LoRA. It never sees the ClippableLinear layers because they're hidden behind the Identity placeholders. This also changes the object hierarchy — `model` is now a `PeftModel` wrapper, which is why the restoration path gets deeper.

  ```python
  model = get_peft_model(model, lora_config)
  ```

* **In step 3 we restore at the correct deep path:** the model is now complete again — the language model has LoRA adapters, and the vision/audio towers are back and intact, frozen (untouched by LoRA). This is exactly what we want: train only the language model's attention layers, let the vision encoder just do its job unchanged.

  ```python
  # Before get_peft_model:  model.model.vision_tower
  # After  get_peft_model:  model.base_model.model.model.vision_tower
  model.base_model.model.model.vision_tower = vision_tower  # put real tower back
  ```



In [ ]:
#####################################
# Install HF Libraries
#####################################

# https://ai.google.dev/gemma/docs
# https://ai.google.dev/gemma/docs/core/gemma_library

# ⚠️ Accelerator: Gemma-4-E4B in 4-bit quantization requires ~8–9 GB VRAM. Use e.g. A100 (40 GB) in Colab

# ⚠️ Gemma 4 support requires very recent transformers version. Don't just use 'pip install transformers'
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q -U accelerate bitsandbytes
!pip install -q -U peft trl datasets # Required for PEFT tuning

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 703.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 52.0 MB/s eta 0:00:00


In [ ]:
# ⚠️ !!! Restart runtime (Runtime → Restart session) and verify version:
import transformers
import torch
from transformers import pipeline, AutoProcessor, Gemma4ForConditionalGeneration, BitsAndBytesConfig, GenerationConfig
print(transformers.__version__)  # Should show something like 5.6.0.dev0

5.6.0.dev0


In [ ]:
#####################################
# Authentification
#####################################

# Use Colab Secrets or GCP Secrets to store Huggingface access token
from google.colab import userdata
from huggingface_hub import login

# Huggingface Token from Colab Secrets
# ⚠️ Create token on https://huggingface.co/settings/tokens and add 'Repositories permissions' for model 'google/gemma-4-E4B-it'
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face
login(token=hf_token)

In [ ]:
#####################################
# Load Model + Processor
#####################################

"""
⚠️ Gemma 4 is multimodal — its vision and audio encoder uses custom 'Gemma4ClippableLinear' wrapper that PEFT doesn't recognize. We need:
   1. Add prepare_model_for_kbit_training — required step before LoRA on any quantized model.
   2. Exclude unused (vision and/or audio) encoder — only finetune the language model layers, or language+vision
"""

import torch
from transformers import AutoProcessor, Gemma4ForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

model_id = "google/gemma-4-E4B-it"

# Define Quantization Configuration to fit model on GPU
  # ⚠️ NEW: BitsAndBytesConfig via quantization_config is now mandatory for HF's Transformers v5+
  # BitsAndBytes shrink raw model from ~30 GB (in float32) to ~8–9 GB (in 4-bit precision) by quantization compression of weights
  # Transformers v5 updated weight-loading pipeline: 'load_in_4bit argument no longer top-level parameter to pass directly into
  # model's initialization. Instead, it must be wrapped in a BitsAndBytesConfig and passed via quantization_config.
# Memory requirements during quantization (rule of thumb):
  # FP16 (standard): 2 bytes per parameter. (14 bytes × 2 = 28 GB)
  # 4-bit (quantized): approximately 0.7 to 0.8 bytes per parameter (including overhead). (14 bytes × 0.75 ≈ 10.5 GB)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Compresses weights from 16/32-bit → 4-bit (quantization) to fit 8.6 GB model on GPU
    bnb_4bit_compute_dtype=torch.bfloat16,  # Precision for calculations: Upcast back to bfloat16 during math.
    bnb_4bit_quant_type="nf4",              # Quantization algorithm (NormalFloat4): Use NF4 format, best for normally-distributed LLM weights.
    bnb_4bit_use_double_quant=True          # Quantize quantization constants: adds second compression to save ~0.4 bits extra per parameter.
)

# Load new Processor: lightweight, handles text+image input preparation
processor = AutoProcessor.from_pretrained(model_id)

# Load Model (without LoRA applied yet)
model = Gemma4ForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    quantization_config=quant_config, # Weights read from disk, then further quantized to 4-bit  NF4 format (~8–9 GB) during the GPU load.
    device_map="auto"                 # GPU placement: weights loaded into VRAM, quantized on-the-fly to 4-bit NF4 format: 'Loading weights: x/y'
)

# NOTE: prepare_model_for_kbit_training is called in the LoRA application cell below on clean model
# after the Identity swap. We don't need to call it here anymore.
# model = prepare_model_for_kbit_training(model)

# First check exact layer names on this clean model
for name, _ in model.named_modules():
    if "q_proj" in name and "vision" not in name and "audio" not in name:
        print(name)
        break  # Just print first match to confirm the path

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

model.language_model.layers.0.self_attn.q_proj


In [ ]:
#####################################
# Check Gemma model details
#####################################

# ⚠️ We need to identify the exact structure of the Gemma model to exclude the vision and audio towers for PEFT tuning

print("\nPrint all named modules to see the exact structure:\n" + 40*"-")
for name, module in model.named_modules():
    if "q_proj" in name:
        print(name)

print("\nTop-level submodules of the model:\n" + 40*"-")
for name, module in model.named_children():
    print(name, type(module).__name__)

print("\nPaths of modality parts:\n" + 40*"-")
for name, module in model.model.named_children():
    print(name, type(module).__name__)


Print all named modules to see the exact structure:
----------------------------------------
model.language_model.layers.0.self_attn.q_proj
model.language_model.layers.1.self_attn.q_proj
model.language_model.layers.2.self_attn.q_proj
model.language_model.layers.3.self_attn.q_proj
model.language_model.layers.4.self_attn.q_proj
model.language_model.layers.5.self_attn.q_proj
model.language_model.layers.6.self_attn.q_proj
model.language_model.layers.7.self_attn.q_proj
model.language_model.layers.8.self_attn.q_proj
model.language_model.layers.9.self_attn.q_proj
model.language_model.layers.10.self_attn.q_proj
model.language_model.layers.11.self_attn.q_proj
model.language_model.layers.12.self_attn.q_proj
model.language_model.layers.13.self_attn.q_proj
model.language_model.layers.14.self_attn.q_proj
model.language_model.layers.15.self_attn.q_proj
model.language_model.layers.16.self_attn.q_proj
model.language_model.layers.17.self_attn.q_proj
model.language_model.layers.18.self_attn.q_proj
mode

From `named_modules()` output we can now see Gemma 4's three towers including the path is `model.language_model.layers.N`:

```
model.model.language_model.layers.0.        ← ✅ train these (text LLM)
model.model.vision_tower.encoder.layers.0.  ← ❌ skip (image encoder, Gemma4ClippableLinear)
model.model.audio_tower.layers.0.           ← ❌ skip (audio encoder, also ClippableLinear)
```

By specifying full path `language_model.layers.*`, PEFT will only match the language model layers and naturally skip both the vision and audio towers entirely. We can see that correct path is `model.model.language_model`. The structure is now clear:

```
model                           ← Gemma4ForConditionalGeneration
└── model.model                 ← Gemma4Model
    ├── model.model.language_model  ← ✅ Gemma4TextModel — we apply LoRA here
    ├── model.model.vision_tower    ← ❌ skip
    ├── model.model.audio_tower     ← ❌ skip
    ├── model.model.embed_vision    ← ❌ skip
    └── model.model.embed_audio     ← ❌ skip
```

The object hierarchy after `get_peft_model()` is:
```
* Before get_peft_model:  model  →  model.model  →  vision_tower
* After  get_peft_model:  model (PeftModel)  →  model.base_model  →   model.base_model.model  →  model.base_model.model.model  →  vision_tower
```

get_peft_model() wraps the model in an extra layer. So when we then do `model.model.vision_tower = vision_tower`, we are attaching it to the PEFT wrapper, not to the underlying model where the actual forward pass runs. The key rule to remember for this workaround: `model.base_model` only comes into existence on the line `model = get_peft_model(...)`, so any code using that path must come after it. And anything that needs the Identity swap (so PEFT doesn't see the real towers) must come before it.


In [ ]:
#####################################
# LoRA Configuration (train only ~1% of parameters)
#####################################

import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


"""
# ⚠️ The cleanest approach here would this be this:

# with correct path: model.model.language_model (not model.language_model)
model.model.language_model = prepare_model_for_kbit_training(model.model.language_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Safe to use short names now — only Gemma4TextModel in scope
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    # etc... more configurations
)

# Wrap only the language model submodule
model.model.language_model = get_peft_model(model.model.language_model, lora_config)
model.model.language_model.print_trainable_parameters()

# ⚠️ -> However we can not use this approach. It will lead to an error message.
  # Good news: LoRA applies successfully! But there is a new error: wrapping the submodule breaks the generation interface that
  # lives on the top-level model.
# As discussed, the cleanest workaround is to temporarily swap out vision/audio towers, apply LoRA to full model, then restore them.

# The core problem: Gemma 4 is so new (released weeks ago) that PEFT hasn't added native support for it yet.
  # This swap trick is the practical workaround until PEFT officially supports Gemma4ClippableLinear.
  # The restored towers are frozen (not touched by LoRA) so training will correctly only update the language model attention layers.
"""


# Step 1: Save the real towers, then replace with Identity placeholders.
  # ⚠️ This must happen before prepare_model_for_kbit_training and get_peft_model,
  # so PEFT never encounters the unsupported Gemma4ClippableLinear layers.
vision_tower = model.model.vision_tower
audio_tower  = model.model.audio_tower
embed_vision = model.model.embed_vision
embed_audio  = model.model.embed_audio

model.model.vision_tower = torch.nn.Identity()
model.model.audio_tower  = torch.nn.Identity()
model.model.embed_vision = torch.nn.Identity()
model.model.embed_audio  = torch.nn.Identity()

# Step 2: Prepare for quantized training.
  # Required for any 4-bit quantized model before applying LoRA.
  # Enables gradient checkpointing and casts non-quantized layers correctly.
model = prepare_model_for_kbit_training(model)

# Step 3: Define LoRA config and wrap the full model.
  # Because the towers are Identity placeholders right now, PEFT only sees
  # the standard language model layers and applies LoRA only there.

"""
Normally we would use 'exclude_modules' or Regex, but this does not work yet. 😭😭😭 So we use the Identity-swap instead.
But let me explain briefly what each is, what didn't work and what could be possible reasons.

1) 'exclude_modules'
  PEFT's type check runs before the exclusion filter. When PEFT scans the model to apply LoRA, the order of operations is roughly:
    a) Walk every module in the model.
    b) For each module, check if its type is one PEFT supports (nn.Linear, nn.Embedding, nn.Conv1d/2d/3d, nn.MultiheadAttention).
    c) If yes, then check whether the module name matches target_modules and doesn't match exclude_modules.
    d) Wrap accordingly.
  The problem: step 2 fails on Gemma 4's vision and audio towers because Gemma4ClippableLinear inherits from nn.Module, not nn.Linear.
  PEFT hits one of those layers, sees an unsupported type, and raises a ValueError immediately — it never gets to step 3 where exclude_modules
  would have skipped it. This was confirmed in the HF discussion thread where Benjamin Bossan (the PEFT maintainer) acknowledged the type check
  happens first: PEFT checks the type of every target module before applying LoRA. Since Gemma4ClippableLinear isn't nn.Linear, PEFT rejects it
  — even though we only want to apply LoRA to the text decoder layers, not the vision encoder. The exclude_modules parameter doesn't help either.
  PEFT runs the type check before filtering, so excluded modules still need to be recognized types.

  So 'exclude_modules' is a real PEFT feature and works fine on normal models, but on Gemma 4 it's useless because the model crashes during
  the type-check pass before exclusion ever gets considered. We can't exclude something PEFT refuses to even look at.

  Our solution with the Identity-swap sidesteps this entirely. By replacing the towers with torch.nn.Identity() before PEFT scans,
  we give PEFT a recognized type at those positions, and Identity is a standard PyTorch primitive. PEFT walks past them without complaint
  (Identity doesn't match q_proj or any other target name, so nothing gets wrapped there), and then we restore the real towers afterward.
  No type check failure, no need for exclude_modules, no need for any filtering at all. The towers are simply invisible to PEFT during the
  critical scan window.


2) Regex:
  In PEFT's LoraConfig, the target_modules parameter normally takes a list of leaf module names like ["q_proj", "k_proj", "v_proj", ...].
  PEFT then walks the model and wraps any module whose leaf name matches one of those strings, regardless of where it lives in module tree.
  That's why on Gemma 4 it ends up wrapping q_proj everywhere it finds one: in the language model, in the vision tower, and in audio tower.
  The leaf-name match is path-blind. The "regex approach" is an alternative: instead of a list of leaf names, we can pass 'target_modules'
  a single regex string, and PEFT will match it against the full dotted path of each module rather than just the leaf name.
  So we can write a pattern that pins LoRA to specific subtrees:

  > target_modules=r".*language_model\.layers\.\d+\.(self_attn|mlp)\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"

  That regex says: "match any module whose full path contains language_model.layers.<some number>.self_attn.<one of these projection names>
  or ...mlp.<one of these projection names>, and ends there." In theory, this gives us a positive whitelist that only matches the language
  model's projection layers and nothing else — no need for exclude_modules at all, and no risk of accidentally wrapping the vision or audio towers.
  It's a PEFT feature and it does work on most models. The HF docs and several PEFT examples recommend it for exactly this kind of
  "I only want LoRA on a specific subtree of a multi-tower model" scenario.

  But when I tried it on our Gemma 4, diagnostic still showed following which means PEFT wrapped LoRA into all three towers regardless of regex.
  * audio_tower LoRA params: 72
  * vision_tower LoRA params: 224
  * language_model LoRA params: 588

  There are some possible reasons:
  1. My version of PEFT may not honor the regex form for this model class. - PEFT's matching logic has changed across versions, and how it interacts
     with quantized 4-bit models, nested PeftModel wrappers, and the specific module hierarchy of Gemma4ForConditionalGeneration is not something
     the PEFT maintainers have explicitly tested. There are open issues on the PEFT GitHub tracker about regex target_modules not behaving as documented in some configurations.
  2. The trainer-mediated wrapping path may bypass the regex - When we pass peft_config=lora_config to SFTTrainer, the trainer calls get_peft_model internally,
     and there's a chance that some intermediate step normalizes or rewrites the target_modules field before it reaches PEFT's actual matching logic.
     This is speculation — I don't know for certain — but it would explain why the regex was silently ignored.
  3. Gemma4ClippableLinear may interact weirdly with PEFT's path resolution - Even with the monkey-patch making the class inherit from nn.Linear, the inner .linear child means
     the actual leaf modules have paths like vision_tower.encoder.layers.0.self_attn.q_proj.linear rather than ...q_proj. The regex I wrote ended in q_proj$ (anchored at end of string),
     which wouldn't match q_proj.linear. So my regex would have failed to match the right layers either, and the wrapping you did see was from PEFT falling back to
     leaf-name matching on q_proj. This is actually the most likely root cause.

"""
lora_config = LoraConfig(
    r=16,                 # LoRA rank — higher = more capacity but more memory
    lora_alpha=32,        # Scaling factor (usually 2x rank)
    lora_dropout=0.05,    # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM",

    # Target modules to tune (attention layers + MLP layers for deeper memory changes)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", # Attention layers
        #"gate_proj", "up_proj", "down_proj"    # MLP layers (allows more capacity/ expressivity, but requires more VRAM)
        ],

    # ⚠️ A more direct approach is to use 'exclude_modules', but this does not work yet either! 😭😭😭
    # exclude_modules=["vision_tower", "multi_modal_projector", "audio_tower", "embed_vision", "embed_audio"],

    # ⚠️ Another approach would be using Regex, but this also does not work! 😭😭😭
    # target_modules=r".*language_model\.layers\.\d+\.(self_attn|mlp)\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"

    # Optional:
    # modules_to_save=["lm_head", "embed_tokens"],
)

  # get_peft_model() wraps `model` inside a new PeftModel layer.
  # The hierarchy becomes:
  #   model (PeftModel)
  #     └─ model.base_model (LoraModel)
  #          └─ model.base_model.model (Gemma4ForConditionalGeneration)
  #               └─ model.base_model.model.model (Gemma4Model)  ← towers live here
model = get_peft_model(model, lora_config)

# Step 4: Restore the real towers — must use the deep path.
  # ⚠️ model.base_model only exists after get_peft_model() above.
  # Using model.model.X here would just set an attribute on the PeftModel
  # wrapper and be silently ignored by the forward pass.
model.base_model.model.model.vision_tower = vision_tower
model.base_model.model.model.audio_tower  = audio_tower
model.base_model.model.model.embed_vision = embed_vision
model.base_model.model.model.embed_audio  = embed_audio

model.print_trainable_parameters()
# Output with attention layers: trainable params: ~10.9M || all params: ~8.0B || trainable%: ~0.14%
# Output including MLP layers:  trainable params: 36,700,160 || all params: 8,032,856,608 || trainable%: 0.4569

trainable params: 10,895,360 || all params: 8,007,051,808 || trainable%: 0.1361


In [ ]:
#####################################
# Prepare Vision/Text Dataset
#####################################

# We take the same sample Amazon product dataset (philschmid/amazon-product-descriptions-vlm) that was used
# in the previous Gemma 3 tutorial: https://ai.google.dev/gemma/docs/core/huggingface_vision_finetune_qlora

from datasets import load_dataset
from PIL import Image

# System message for the assistant
system_message = "You are an expert product description writer for Amazon."

# User prompt that combines the user query and the schema
user_prompt = """Create a Short Product description based on the provided <PRODUCT> and <CATEGORY> and image.
Only return description. The description should be SEO optimized and for a better mobile search experience.

<PRODUCT>
{product}
</PRODUCT>

<CATEGORY>
{category}
</CATEGORY>
"""

# Convert dataset to OAI messages
def format_data(sample):
    return {
        "messages": [
            {
                "role": "system",

                # For Gemma 4 this might work either way, but to be safe and consistent with how the collator processes it
                # we switch back from list-of-dicts format back to this structured format.
                "content": [{"type": "text", "text": system_message}],
                #"content": system_message,
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": user_prompt.format(
                            product=sample["Product Name"],
                            category=sample["Category"],
                        ),
                    },
                    {
                        "type": "image",
                        "image": sample["image"],
                    },
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": sample["description"]}],
            },
        ],
    }

def process_vision_info(messages: list[dict]) -> list[Image.Image]:
    image_inputs = []
    # Iterate through each conversation
    for msg in messages:
        # Get content (ensure it's a list)
        content = msg.get("content", [])
        if not isinstance(content, list):
            content = [content]

        # Check each content element for images
        for element in content:
            if isinstance(element, dict) and (
                "image" in element or element.get("type") == "image"
            ):
                # Get the image and convert to RGB
                if "image" in element:
                    image = element["image"]
                else:
                    image = element
                image_inputs.append(image.convert("RGB"))
    return image_inputs

# Load dataset from the hub
dataset = load_dataset("philschmid/amazon-product-descriptions-vlm", split="train")
dataset = dataset.train_test_split(test_size=0.1)

# Convert dataset to OAI messages
# need to use list comprehension to keep Pil.Image type, .mape convert image to bytes
dataset_train = [format_data(sample) for sample in dataset["train"]]
dataset_test = [format_data(sample) for sample in dataset["test"]]

print(dataset_train[345]["messages"])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/47.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1345 [00:00<?, ? examples/s]

[{'role': 'system', 'content': [{'type': 'text', 'text': 'You are an expert product description writer for Amazon.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Create a Short Product description based on the provided <PRODUCT> and <CATEGORY> and image.\nOnly return description. The description should be SEO optimized and for a better mobile search experience.\n\n<PRODUCT>\nFunko 32654 Pop Pez: Nightmare Before Christmas - Jack Skellington, Multicolor\n</PRODUCT>\n\n<CATEGORY>\nNone\n</CATEGORY>\n'}, {'type': 'image', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=269x500 at 0x78BF8124E360>}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': "Bring home the Pumpkin King! This Funko Pop! Pez dispenser features Jack Skellington from Tim Burton's The Nightmare Before Christmas.  A collectible must-have for fans, this multicolor vinyl figure is perfect for displaying or dispensing your favorite treats.  Jack Skellington Pop! Pez Dispenser."}]}]


In [ ]:
#####################################
# Prepare PEFT Tuning (Training)
#####################################

# If we want to run a quick run — use only 50 samples instead of the full dataset
#dataset_train = [format_data(sample) for sample in dataset["train"].select(range(50))]
#dataset_test  = [format_data(sample) for sample in dataset["test"].select(range(10))]

# Go back to the full dataset
#dataset_train = [format_data(sample) for sample in dataset["train"]]  # ~1,200 samples
#dataset_test  = [format_data(sample) for sample in dataset["test"]]   # ~130 samples

# We need to add a 'collate_fn'
  #  Because images are live PIL objects in the list, we can't pass them as a dataset_text_field.
  # Instead, the collator calls processor.apply_chat_template() + processor() on each batch at training time:
def collate_fn(examples):
    texts = []
    images = []
    for example in examples:
        # Extract PIL images from the messages structure
        image_inputs = process_vision_info(example["messages"])
        # Render the chat template to a string (no tokenization yet)
        text = processor.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
        images.extend(image_inputs)

    # Now process text + images together into model inputs
    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
        add_special_tokens=False  # apply_chat_template already adds <bos>
    )
    # Labels = input_ids, but with padding tokens masked to -100
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    return batch

# Training Configuration
sft_config = SFTConfig(
    output_dir="./gemma4-finetuned",
    num_train_epochs=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,            # ← was 2e-4, reduce by 4x to stop oscillation
    lr_scheduler_type="cosine",    # ← gradually decays LR as training progresses

    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    # ↓ These two are the critical vision-specific additions:
    dataset_text_field="",                           # dummy — collator handles this
    dataset_kwargs={"skip_prepare_dataset": True},   # don't pre-tokenize the dataset
)
sft_config.remove_unused_columns = False  # must be set AFTER construction

# Train
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,       # optional but you have it
    processing_class=processor,
    #peft_config=lora_config,        # pass LoRA config here instead of pre-applying it
    data_collator=collate_fn,        # ← new
)

In [ ]:
#####################################
# Double-check that Audio+Vision Tower are excluded
#####################################

"""
If audio_lora > 0 and vision_lora == 0, that confirms exclude_modules is partially working.
The simplest robust fix is to use a regex target_modules that explicitly only matches language model layers,
so exclusion isn't even needed:
"""

audio_lora = sum(1 for n,_ in trainer.model.named_parameters()
                 if "audio_tower" in n and "lora_" in n)
vision_lora = sum(1 for n,_ in trainer.model.named_parameters()
                  if "vision_tower" in n and "lora_" in n)
lang_lora = sum(1 for n,_ in trainer.model.named_parameters()
                if "language_model" in n and "lora_" in n)
print(f"audio_tower LoRA params: {audio_lora}")
print(f"vision_tower LoRA params: {vision_lora}")
print(f"language_model LoRA params: {lang_lora}")
# ⚠️ Output should show only params in language_model, and 0 LoRA params for audio_tower and vision_tower

audio_tower LoRA params: 0
vision_tower LoRA params: 0
language_model LoRA params: 336


In [ ]:
#####################################
# Run PEFT Tuning (Training)
#####################################

trainer.train()

# Save LoRA Adapter (saves only small LoRA weights (~50 MB), not the full 8 GB model)
model.save_pretrained("./gemma4-lora-adapter")
processor.save_pretrained("./gemma4-lora-adapter")

# Steps are calculated: total steps = (dataset_size / batch_size / gradient_accumulation) × epochs
  # dataset_size            ≈ 1,212 samples  (train split, ~90% of dataset)
  # per_device_batch_size   = 1
  # gradient_accumulation   = 4              (effective batch = 4)
  # epochs                  = 4
  # total steps = (1212 / 1 / 4) × 4 = 303 × 4 = 1,212 (with one epoch = 303 steps)

# ⚠️ We see the warning on top (`tokenizer has new PAD/BOS/EOS tokens`). This is harmless. It just means Gemma 4's chat template
  # added new special tokens that were not in the base tokenizer config. Nothing we need to fix there.

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
5,82.209094
10,56.338629
15,42.152716
20,36.604562
25,33.131207
30,30.271259
35,27.597736
40,24.254370
45,20.256061
50,16.705351


Step,Training Loss
5,82.209094
10,56.338629
15,42.152716
20,36.604562
25,33.131207
30,30.271259
35,27.597736
40,24.254370
45,20.256061
50,16.705351


['./gemma4-lora-adapter/processor_config.json']

In [ ]:
#####################################
# Connect PEFT layer with Base Model
#####################################

from peft import PeftModel

"""
# ⚠️ Normally this would be the clean approach, once PEFT has been updated to natively recognize Gemma4ClippableLinear
# But this doesnt work now yet. Just keep it for future, when we can skip the workaround.

base_model = Gemma4ForConditionalGeneration.from_pretrained(
    model_id, quantization_config=quant_config, device_map="auto"
)
model = PeftModel.from_pretrained(base_model, "./gemma4-lora-adapter")
"""

# the workaround:

# 1. Load the base model
  # This instantiates the full Gemma 4 architecture into memory (applying the specified bitsandbytes quantization).
  # The resulting base_model object contains a nested hierarchy of PyTorch modules, including the language model blocks,
  # the vision encoder (vision_tower), and the audio encoder (audio_tower).
base_model = Gemma4ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

# 2. Module Swapping (The Bypass): Save and temporarily replace vision/audio towers with Identity placeholders
 # We extract the memory references to the multimodal sub-modules and store them in temporary Python variables.
 # Then, we overwrite those specific attributes in the base_model's computational graph with torch.nn.Identity().
 # This works because torch.nn.Identity() is a primitive PyTorch module that simply returns its input unchanged.
 # It is standard, recognized layer type. By mutating the graph in this way, we temporarily remove the unsupported
 # ClippableLinear modules from the module tree before PEFT can scan them.
vision_tower = base_model.model.vision_tower
audio_tower = base_model.model.audio_tower
embed_vision = base_model.model.embed_vision
embed_audio = base_model.model.embed_audio

base_model.model.vision_tower = torch.nn.Identity()
base_model.model.audio_tower = torch.nn.Identity()
base_model.model.embed_vision = torch.nn.Identity()
base_model.model.embed_audio = torch.nn.Identity()

# 3. Injecting the PEFT Adapter - Load the PEFT adapter (PEFT will no longer see the ClippableLinear layers)
  # PEFT initiates its module traversal on the modified base_model. It scans the graph, encounters the Identity layers,
  # ignores them as benign, and successfully reaches the standard torch.nn.Linear layers inside the language model blocks.
  # Injection Upon finding the target modules specified in the adapter's configuration, PEFT wraps those base weights with
  # its LoRA linear classes, successfully mapping the trained low-rank matrices into the execution path.
  # The returned model is now a PeftModel object, which encapsulates the original model.
model = PeftModel.from_pretrained(base_model, "./gemma4-lora-adapter")

# 4. Restoring the Architecture - Restore the towers to the wrapped model
  # Note: PEFT wraps the base model, so we route the restoration through base_model.model
  # Now that PEFT's initialization constraints have been satisfied, we must repair the forward pass of the multimodal components.
  # Because PeftModel acts as a wrapper, the original base model is now nested under .base_model.
  # The Restoration: We navigate down the new object hierarchy (model.base_model.model.model) to locate the Identity placeholders
  # and overwrite them with the original module references we saved in Step 2.
model.base_model.model.model.vision_tower = vision_tower
model.base_model.model.model.audio_tower = audio_tower
model.base_model.model.model.embed_vision = embed_vision
model.base_model.model.model.embed_audio = embed_audio

# Result: The fully reconstructed Gemma 4 computation graph is restored. The vision and audio encoders retain their
# native ClippableLinear weights, while the language model components now seamlessly execute the LoRA forward pass.

print("PEFT model loaded successfully!")

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

PEFT model loaded successfully!


In [ ]:
#####################################
# Run Inference on Tuned Model (Vision + Text)
#####################################
import requests
from PIL import Image
from io import BytesIO

# ============================================================
# Choose test input — pick ONE of the two options below
# ============================================================

# --- Option A: Use a sample from the test dataset ---
# This guarantees image, product name, and category all match each other,
# and gives us the ground truth description for comparison.
USE_TEST_SAMPLE = True   # set True to use a sample from dataset_test
TEST_INDEX = 0           # only used if USE_TEST_SAMPLE is True, try 0, 5, 10, etc. to see different products

# --- Option B: Use a custom product (image + text triple) ---
# Useful for ad-hoc testing on products outside the dataset.
# Set USE_TEST_SAMPLE = False above to activate this.
custom_product  = "FT008 4-Channel 27mhz High Speed Racing RC Boat - RED"
custom_category = "Toys & Games | Hobbies | Remote & App Controlled Vehicles & Parts | Remote & App Controlled Vehicles | Watercraft"
custom_image_url = "https://m.media-amazon.com/images/I/81+7Up7IWyL._AC_SY300_SX300_.jpg"

# ============================================================
# Build the input
# ============================================================
if USE_TEST_SAMPLE:
    test_sample = dataset_test[TEST_INDEX]
    test_image  = process_vision_info(test_sample["messages"])[0]
    # Pull the user prompt text directly from the sample so image and text match
    user_text   = test_sample["messages"][1]["content"][0]["text"]
    # Ground truth — handle both flat-string and list-of-dicts assistant content
    gt_content   = test_sample["messages"][2]["content"]
    ground_truth = gt_content if isinstance(gt_content, str) else gt_content[0]["text"]
else:
    test_image = Image.open(
        requests.get(custom_image_url, stream=True).raw
    ).convert("RGB")
    user_text = user_prompt.format(product=custom_product, category=custom_category)
    ground_truth = None   # no ground truth for custom inputs

# --- Build messages exactly as during training ---
# Image first, text second — Gemma 4 best practice for multimodal prompts.
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "Create a Short Product description based on the provided <PRODUCT> and <CATEGORY> and image. Only return description. The description should be SEO optimized and for a better mobile search experience."}],
    },
    {
        "role": "user",
        "content": [
            {"type": "text",  "text": user_text},
            {"type": "image", "image": test_image},
        ],
    },
]

# --- Step 1: Render chat template to a string (no tokenization yet) ---
# This mirrors EXACTLY what collate_fn does during training.
# tokenize=False gives us the raw formatted string including all special tokens.
prompt_string = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# --- Step 2: Tokenize text + process image together ---
# IMPORTANT: add_special_tokens=False because apply_chat_template already added <bos>
# The processor handles both: tokenizing the text AND encoding the image into pixel_values
inputs = processor(
    text=prompt_string,
    images=[test_image],          # must be a list, even for a single image
    return_tensors="pt",
    add_special_tokens=False,
).to(model.device)

# --- Step 3: Generate ---
# Stop on EOS *or* the Gemma 4 end-of-turn marker <turn|>, so the model
# can't accidentally start a second turn before max_new_tokens runs out.
stop_token_ids = [
    processor.tokenizer.eos_token_id,
    processor.tokenizer.convert_tokens_to_ids("<turn|>"),
]

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,           # greedy decoding — deterministic, good for testing
        eos_token_id=stop_token_ids,
    )

# --- Step 4: Decode only the newly generated tokens ---
# inputs["input_ids"].shape[-1] is the length of the prompt — we skip those
response = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

# ============================================================
# Display results
# ============================================================
print("=" * 60)
print("PROMPT (user text):")
print(user_text[:300] + ("..." if len(user_text) > 300 else ""))
print("=" * 60)
print("\nFINE-TUNED MODEL OUTPUT:")
print(response)
if ground_truth is not None:
    print("\nGROUND TRUTH (from dataset):")
    print(ground_truth)
print("=" * 60)

PROMPT (user text):
Create a Short Product description based on the provided <PRODUCT> and <CATEGORY> and image.
Only return description. The description should be SEO optimized and for a better mobile search experience.

<PRODUCT>
Hasbro Overwatch Tracer Roleplay Mask with Removable Hair Accessory - Blizzard Video Gam...

FINE-TUNED MODEL OUTPUT:
Unleash your inner Tracer! This Hasbro Overwatch Tracer Roleplay Mask comes with a removable wig accessory for ultimate immersive play. Perfect for Halloween costumes or any roleplay adventure. Shop now for this Blizzard video game character collectible!

GROUND TRUTH (from dataset):
Become Overwatch's Tracer! This Hasbro roleplay mask features a removable hair accessory for ultimate playtime. Perfect for kids who love to dress up and pretend play. Officially licensed Blizzard Entertainment product.


In [ ]:
#####################################
# Run second test
#####################################

test_sample = dataset_test[0]
test_image = process_vision_info(test_sample["messages"])[0]

# Pull the user prompt text directly from the test sample so image,
# product name, and category all match.
user_text = test_sample["messages"][1]["content"][0]["text"]

# Ground truth — handle both flat-string and list-of-dicts assistant content
gt_content = test_sample["messages"][2]["content"]
ground_truth = gt_content if isinstance(gt_content, str) else gt_content[0]["text"]

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "Create a Short Product description based on the provided <PRODUCT> and <CATEGORY> and image. Only return description. The description should be SEO optimized and for a better mobile search experience."}],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": user_text},
            {"type": "image", "image": test_image},
        ],
    },
]

prompt_string = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True,
)
inputs = processor(
    text=prompt_string, images=[test_image],
    return_tensors="pt", add_special_tokens=False,
).to(model.device)

# Fine-tuned output
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
ft_response = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True,
)

# Base model output (LoRA disabled)
with model.disable_adapter():
    with torch.inference_mode():
        base_outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
base_response = processor.decode(
    base_outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True,
)

print("PRODUCT FROM TEST SAMPLE:")
print(user_text[:200], "...\n")
print("FINE-TUNED:", ft_response, "\n")
print("BASE MODEL:", base_response, "\n")
print("GROUND TRUTH:", ground_truth)

PRODUCT FROM TEST SAMPLE:
Create a Short Product description based on the provided <PRODUCT> and <CATEGORY> and image.
Only return description. The description should be SEO optimized and for a better mobile search experience. ...

FINE-TUNED: Unleash your inner Tracer! This Hasbro Overwatch Tracer Roleplay Mask includes a removable wig accessory for immersive Blizzard video game roleplay. Perfect for cosplay or imaginative play. Shop now for your Tracer gear! 

BASE MODEL: Unleash your inner hero with the Hasbro Overwatch Tracer Roleplay Mask! Perfect for fans of Blizzard video games, this mask lets kids jump into action during dress-up and pretend play. High-quality and fun, it's the ultimate gift for any Overwatch enthusiast! #Overwatch #Tracer #RoleplayToys #BlizzardGames 

GROUND TRUTH: Become Overwatch's Tracer! This Hasbro roleplay mask features a removable hair accessory for ultimate playtime. Perfect for kids who love to dress up and pretend play. Officially licensed Blizzard 